# Loading in dependencies

In [1]:
#Dependencies
import numpy as np
import pandas as pd
import requests
import jcamp as jc
import os
from scipy.interpolate import interp1d

# Setting up pandas DataFrames

In [2]:
#Create ultimate dataset with adjacency matrices and spectra information
dataset = pd.DataFrame(columns=['IR Spectra','Bond Adjacency Matrix'])

#Empty list to store spectral data, will later to inserted into the dataframe
spectra = []
matrices = []

#P.J. Linstrom and W.G. Mallard, Eds., NIST Chemistry WebBook, NIST Standard Reference Database Number 69, National Institute of Standards and Technology, Gaithersburg MD, 20899, https://doi.org/10.18434/T4D303, (retrieved June 26, 2022). 
#Species list downloaded from the NIST
species_path = os.getcwd()+'/Datasets/species.txt'

#Load in species list
species = pd.read_csv(species_path,delimiter="\t")
species.columns = ['Species Name', 'Formula', 'CAS']

#Remove all species that are missing data
species_filtered = species.dropna()

#Get series of CAS's to search
CAS = species_filtered['CAS']

# Scraping the NIST database for spectra data and .mol files

In [3]:
#Modify if you want to scrape more or less molecular information
limit = 10 #scraping around 6 hours worth of downloads, at 1 molecule per second entries initially

#List to hold onto molecule information until the adjacency matrix is made
moles=[]

#Index through all CAS's in the filtered list of species
for n,element in enumerate(CAS.head(limit).items()):
    index = element[0]
    compound = element[1]

    #Get IR data url
    IR_url = f'https://webbook.nist.gov/cgi/cbook.cgi?JCAMP={compound}&Index=0&Type=IR'
    
    #Get .mol file url
    mol_url = f'https://webbook.nist.gov/cgi/cbook.cgi?Str2File=C{compound}'
    
    #Make sure that nothing weird ends up in the dataset
    try:
        print()
        #Download JCAMP file
        jdx = requests.get(IR_url,stream=True)

        #Download .mol file
        mol = requests.get(mol_url,stream=True)
        
        #Read spectral data
        content = jdx.content.splitlines()
        content = [line.decode("utf-8") for line in content]
        data = jc.jcamp_read(content)
        
        #Make sure spectrum was found
        if data['title'] == 'Spectrum not found.':
            print("ERROR: Spectrum was not found.")
            continue
        
        #Spectra processing
        jc.JCAMP_calc_xsec(data)

        #In the future maybe focus the model of a region of the IR spectra
        #Fingerprint region contains alot of C-C bonds, so it is critical to the structure of the molecule
        #This is why we have to sacrifice any data that does not document the fingerprint region, leading to about
        #30% data loss
        
        #Range of wavenumbers
        x = np.linspace(600,3500,num=12000) #12000 samples between 500cm^-1 and 3500cm^-1
        
        #Absorbance spectra variable
        absorbanceSpectra = []
        
        #interpolate data between 600cm^-1 and 4000cm^-1 
        if(data['yunits'].lower()=='absorbance'):
            f = interp1d(data['wavenumbers'],data['y'],assume_sorted=True)
            absorbanceSpectra = f(x)

        elif (data['yunits'].lower() == 'transmittance'):
            f = interp1d(data['wavenumbers'],data['absorbance'],assume_sorted=True)
            absorbanceSpectra = f(x)
        else:
            print("Incompatible units")
            continue
        
        #Save the .mol and spectrum data
        moles.append(mol.content.decode())  
        spectra.append(absorbanceSpectra)
        
        print(f"Molecule #{n+1} ({data['title']}) out of {limit}")
    
    #Skip any entries that bring an error
    except Exception as e:
        print("Error")
        print(e)
        continue



Molecule #1 (Benzene, 1-chloro-4-nitro-) out of 10

Molecule #2 (p-Nitroaniline) out of 10

Molecule #3 (4-Nitrophenol) out of 10

ERROR: Spectrum was not found.

ERROR: Spectrum was not found.

Molecule #6 (n-BUTYLTRIMETHYLSILANE) out of 10

Error
A value in x_new is below the interpolation range.

Molecule #8 (Cyclohexanol, 2-(9H-purin-9-yl)-, trans-) out of 10

Error
A value in x_new is below the interpolation range.

Molecule #10 (4'-Methoxyacetophenone) out of 10


# Processing .mol data to extract the bond adjacency matrix

In [4]:
from rdkit import Chem

#Important for matrix padding later
dimensions = []
matrices = []

#Load in the .mol data
for file in moles:
    m = Chem.MolFromMolBlock(file)    
    
    #Catch broken files
    if m is None:
        #add NaN to list, we will later filter out these
        dimensions.append(0)
        matrices.append(np.nan)
        continue

    #m = Chem.AddHs(m,explicitOnly=True) May be a bad idea to omit hydrogens when training model, we'll see
    
    #Create adjacency matrix for atoms
    am = Chem.GetAdjacencyMatrix(m)

    #Create adjacency matrix for bonds
    X,Y = am.shape

    #Encode bond types into the adjacency matrix
    for A in range(X):
        for B in range(Y):
            bond = m.GetBondBetweenAtoms(A,B)

            #Do nothing if a bond does not exist
            if bond != None:                
                #put in the bond type as a float
                am[A,B] = am[A,B]*bond.GetBondTypeAsDouble() #Adjacency matrix with bond types encoded in
    
    dimensions.append(X) #Store the dimension of this matrix, so that we can appropriately pad it later
    matrices.append(am) #Store the adjacency matrix     

# Padding adjacency matrices

In [5]:
#Pad molecule matrices
largestDimension = np.amax(dimensions)

for i in range(len(matrices)):
    matrix = matrices[i]
    #Difference in dimensions, matrix will always be a square so we can just take one side
    try:
        padding = largestDimension-matrix.shape[1]

        matrices[i] = np.pad(matrix,((0,padding),(0,padding)) ).flatten()
    except AttributeError: #Because of the NaN values
        continue

In [6]:
print("Finished padding adjacency matrices")
print(f'Matrices length: {len(matrices)}, Matrices shape: {matrices[0].shape}')
print(f'Spectrum length: {len(spectra)}, Spectra shape: {spectra[0].shape}')

Finished padding adjacency matrices
Matrices length: 6, Matrices shape: (256,)
Spectrum length: 6, Spectra shape: (12000,)


# Assembling the DataFrame

In [7]:
#Insert our columns of data
dataset['IR Spectra'] = spectra
dataset['Bond Adjacency Matrix'] = matrices

# Packing it up for storage

In [8]:
dataset.to_csv('dataset', sep='\t')

In [9]:
print(dataset)

                                          IR Spectra  \
0  [0.01629807625000003, 0.016552774695495466, 0....   
1  [0.010533840949739498, 0.010572181063798224, 0...   
2  [0.0003479999999999994, 0.00035269839153262677...   
3  [0.06024344731275887, 0.06179269982334457, 0.0...   
4  [0.05929056299052146, 0.05758668842457634, 0.0...   
5  [0.002276050000000003, 0.0022488711100925127, ...   

                               Bond Adjacency Matrix  
0  [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
1  [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
4  [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
5  [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
